# 🛢️ Smart Decline Curve Analysis (DCA) Dashboard
### Classical Petroleum Engineering + Data Engineering + AI

This notebook builds a complete, self-contained project:

1. **Simulated raw field data ingestion** (mimics a multi-well SCADA/production export)
2. **Data Engineering: Bronze → Silver → Gold ETL pipeline** in DuckDB (validation, cleaning, gap-filling, feature engineering)
3. **Classical Decline Curve Analysis** — Arps hyperbolic decline fit (the industry-standard technique)
4. **AI Layer 1** — a global XGBoost model trained across all wells, benchmarked against classical Arps on a holdout window
5. **AI Layer 2** — Isolation Forest anomaly detection to flag wells deviating from their natural decline (mechanical issues, water breakthrough, etc.)
6. **A polished Streamlit dashboard** to explore it all interactively, launched directly from Colab via `pyngrok`

Just run the cells top to bottom. Everything (data, warehouse, model, app) is created in this Colab session — no external files needed.


## 0. Install dependencies

In [1]:
!pip install -q duckdb xgboost scikit-learn plotly streamlit pyngrok scipy pandas numpy
print("All packages installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 54.2 MB/s eta 0:00:00
All packages installed.


## 1. Simulate raw field data (Bronze layer source)

In a real project this would be a SCADA export, a WITSML feed, or a production accounting system dump.
Here we simulate 15 wells over ~2.5 years with realistic Arps-style decline, water cut rise, GOR trends,
random shut-in periods, and **intentionally injected bad readings** (negative rates, sensor spikes, missing days)
so the ETL step below has real cleaning work to do — just like a real field dataset.

In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

def arps_rate(t, qi, di, b):
    """Arps hyperbolic decline. b=0 -> exponential, b=1 -> harmonic, 0<b<1 -> hyperbolic."""
    if b == 0:
        return qi * np.exp(-di * t)
    return qi / np.power(1 + b * di * t, 1.0 / b)

N_WELLS = 15
DAYS = 900
start_date = datetime(2022, 1, 1)

records = []
well_meta = []

for i in range(N_WELLS):
    well_id = f"WELL-{i+1:02d}"
    qi = np.random.uniform(300, 1400)          # initial oil rate, bbl/d
    di = np.random.uniform(0.002, 0.010)        # nominal daily decline
    b = np.random.uniform(0.1, 0.9)             # hyperbolic exponent
    gor_base = np.random.uniform(500, 1200)     # scf/bbl
    wc_base = np.random.uniform(0.02, 0.10)     # starting water cut
    wc_rise = np.random.uniform(0.0002, 0.0009) # water cut rise per day
    choke = np.random.choice([24, 32, 40, 48])
    lat, lon = np.random.uniform(29.0, 31.5), np.random.uniform(47.5, 49.5)

    well_meta.append({
        "well_id": well_id, "qi": qi, "di": di, "b": b,
        "choke_size_64th": choke, "latitude": lat, "longitude": lon,
        "spud_date": (start_date - timedelta(days=int(np.random.uniform(30, 400)))).date().isoformat()
    })

    # a couple of random shut-in windows per well
    shutins = sorted(np.random.choice(range(60, DAYS - 60), size=np.random.randint(1, 3), replace=False))
    shutin_windows = [(s, s + np.random.randint(3, 15)) for s in shutins]

    for t in range(DAYS):
        date = start_date + timedelta(days=t)
        true_rate = arps_rate(t, qi, di, b)

        is_shutin = any(s <= t <= e for s, e in shutin_windows)
        noise = np.random.normal(0, 0.05 * true_rate)
        oil_rate = 0.0 if is_shutin else max(true_rate + noise, 0)

        # occasionally inject a bad sensor reading (negative / spike) -> for the ETL cleaning step to catch
        if np.random.rand() < 0.01 and not is_shutin:
            oil_rate = -abs(oil_rate) * np.random.uniform(0.1, 0.5)
        if np.random.rand() < 0.005 and not is_shutin:
            oil_rate = oil_rate * np.random.uniform(3, 5)

        water_cut = min(wc_base + wc_rise * t + np.random.normal(0, 0.01), 0.98)
        gor = gor_base * (1 + 0.15 * (t / DAYS)) + np.random.normal(0, 30)
        gas_rate = oil_rate * gor / 1000.0  # mscf/d
        water_rate = oil_rate * water_cut / max(1 - water_cut, 1e-3) if oil_rate > 0 else 0.0
        whp = np.random.uniform(150, 400) * (1 - 0.3 * (t / DAYS)) if oil_rate > 0 else 0.0

        # randomly drop ~1% of rows to simulate missing telemetry
        if np.random.rand() < 0.01:
            continue

        records.append({
            "well_id": well_id,
            "date": date.date().isoformat(),
            "oil_rate_bbl": round(oil_rate, 2),
            "gas_rate_mscf": round(gas_rate, 2),
            "water_rate_bbl": round(water_rate, 2),
            "water_cut": round(water_cut, 4),
            "gor_scf_bbl": round(gor, 1),
            "wellhead_pressure_psi": round(whp, 1),
            "choke_size_64th": choke,
        })

raw_df = pd.DataFrame(records)
meta_df = pd.DataFrame(well_meta)

raw_df.to_csv("raw_scada_export.csv", index=False)
meta_df.to_csv("well_metadata.csv", index=False)

print(f"Generated {len(raw_df):,} raw production records across {N_WELLS} wells")
print(f"Bad readings injected (negative or spikes) simulate real SCADA noise for the ETL step to clean.")
raw_df.head()


Generated 13,369 raw production records across 15 wells
Bad readings injected (negative or spikes) simulate real SCADA noise for the ETL step to clean.


,well_id,date,oil_rate_bbl,gas_rate_mscf,water_rate_bbl,water_cut,gor_scf_bbl,wellhead_pressure_psi,choke_size_64th
0,WELL-01,2022-01-01,663.38,604.99,25.86,0.0375,912.0,287.2,40
1,WELL-01,2022-01-02,702.50,628.50,21.01,0.0290,894.7,230.3,40
2,WELL-01,2022-01-03,767.14,709.38,24.65,0.0311,924.7,284.0,40
3,WELL-01,2022-01-04,693.20,642.66,37.42,0.0512,927.1,165.4,40
4,WELL-01,2022-01-05,699.21,618.51,18.57,0.0259,884.6,176.0,40


## 2. Data Engineering — Bronze → Silver → Gold ETL pipeline (DuckDB)

- **Bronze**: raw ingestion, untouched
- **Silver**: validated & cleaned — negative/spike readings nulled and interpolated, calendar gaps filled per well
- **Gold**: analysis-ready, feature-engineered table (rolling rates, decline rate %, water cut/GOR slopes, lag features, cumulative production)

This is the layer most "DCA in Excel" projects skip entirely — here it's a proper warehouse anyone on a team could query.

In [3]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect("dca_warehouse.duckdb")

# ---------- BRONZE: raw ingestion, as-is ----------
raw_df = pd.read_csv("raw_scada_export.csv", parse_dates=["date"])
meta_df = pd.read_csv("well_metadata.csv")

con.execute("CREATE OR REPLACE TABLE bronze_production AS SELECT * FROM raw_df")
con.execute("CREATE OR REPLACE TABLE well_metadata AS SELECT * FROM meta_df")
print(f"BRONZE  | bronze_production: {len(raw_df):,} rows (raw, unvalidated)")

# ---------- SILVER: validated & cleaned ----------
df = raw_df.copy().sort_values(["well_id", "date"])

n_before = len(df)
neg_mask = df["oil_rate_bbl"] < 0
df.loc[neg_mask, "oil_rate_bbl"] = np.nan          # negative rates are sensor errors -> null out

# spike detection: flag readings > 3x the well's rolling median as outliers -> null out
df["rolling_median"] = df.groupby("well_id")["oil_rate_bbl"].transform(
    lambda s: s.rolling(14, min_periods=3, center=True).median()
)
spike_mask = (df["oil_rate_bbl"] > 3 * df["rolling_median"]) & df["rolling_median"].notna()
df.loc[spike_mask, "oil_rate_bbl"] = np.nan

# interpolate short gaps (nulled/missing values) per well, cap at 7-day gaps
df["oil_rate_bbl"] = df.groupby("well_id")["oil_rate_bbl"].transform(
    lambda s: s.interpolate(method="linear", limit=7, limit_direction="both")
)

# fill any remaining calendar gaps per well (reindex to daily) so the time series is continuous
filled = []
for wid, g in df.groupby("well_id"):
    g = g.set_index("date").sort_index()
    full_idx = pd.date_range(g.index.min(), g.index.max(), freq="D")
    g = g.reindex(full_idx)
    g["well_id"] = wid
    g["oil_rate_bbl"] = g["oil_rate_bbl"].interpolate(limit=7, limit_direction="both")
    for col in ["gas_rate_mscf", "water_rate_bbl", "water_cut", "gor_scf_bbl",
                "wellhead_pressure_psi", "choke_size_64th"]:
        g[col] = g[col].interpolate(limit=7, limit_direction="both").ffill().bfill()
    g["is_shutin"] = g["oil_rate_bbl"].fillna(0) <= 1.0
    g["was_imputed"] = g["oil_rate_bbl"].isna()
    g["oil_rate_bbl"] = g["oil_rate_bbl"].fillna(0)
    g.index.name = "date"
    filled.append(g.reset_index())

silver_df = pd.concat(filled, ignore_index=True)
n_cleaned = neg_mask.sum() + spike_mask.sum()
con.execute("CREATE OR REPLACE TABLE silver_production AS SELECT * FROM silver_df")

print(f"SILVER  | silver_production: {len(silver_df):,} rows")
print(f"        | negative readings nulled: {neg_mask.sum()}  | spike readings nulled: {spike_mask.sum()}")
print(f"        | calendar gaps filled via interpolation: {silver_df['was_imputed'].sum()}")

# ---------- GOLD: feature-engineered, analysis-ready ----------
def engineer_features(g):
    g = g.sort_values("date").reset_index(drop=True)
    g["days_on_prod"] = np.arange(len(g))
    g["cum_oil_bbl"] = g["oil_rate_bbl"].cumsum()
    g["rate_7d_avg"] = g["oil_rate_bbl"].rolling(7, min_periods=1).mean()
    g["rate_30d_avg"] = g["oil_rate_bbl"].rolling(30, min_periods=1).mean()
    g["decline_rate_pct"] = g["rate_7d_avg"].pct_change(periods=30) * -100
    g["decline_rate_pct"] = g["decline_rate_pct"].replace([np.inf, -np.inf], np.nan)
    g["water_cut_slope_30d"] = g["water_cut"].diff(30) / 30
    g["gor_slope_30d"] = g["gor_scf_bbl"].diff(30) / 30
    g["oil_rate_lag7"] = g["oil_rate_bbl"].shift(7)
    g["oil_rate_lag14"] = g["oil_rate_bbl"].shift(14)
    g["oil_rate_lag30"] = g["oil_rate_bbl"].shift(30)
    return g

gold_frames = [engineer_features(g) for _, g in silver_df.groupby("well_id")]
gold_df = pd.concat(gold_frames, ignore_index=True)
gold_df = gold_df.merge(meta_df[["well_id", "qi", "di", "b"]], on="well_id", how="left")

con.execute("CREATE OR REPLACE TABLE gold_features AS SELECT * FROM gold_df")
print(f"GOLD    | gold_features: {len(gold_df):,} rows, {gold_df.shape[1]} columns (analysis-ready)")

con.close()
gold_df.head()


BRONZE  | bronze_production: 13,369 rows (raw, unvalidated)
SILVER  | silver_production: 13,500 rows
        | negative readings nulled: 150  | spike readings nulled: 54
        | calendar gaps filled via interpolation: 0
GOLD    | gold_features: 13,500 rows, 25 columns (analysis-ready)


,date,well_id,oil_rate_bbl,gas_rate_mscf,water_rate_bbl,water_cut,gor_scf_bbl,wellhead_pressure_psi,choke_size_64th,rolling_median,...,rate_30d_avg,decline_rate_pct,water_cut_slope_30d,gor_slope_30d,oil_rate_lag7,oil_rate_lag14,oil_rate_lag30,qi,di,b
0,2022-01-01,WELL-01,663.38,604.99,25.86,0.0375,912.0,287.2,40.0,699.210,...,663.380000,NaN,NaN,NaN,NaN,NaN,NaN,711.994131,0.009606,0.685595
1,2022-01-02,WELL-01,702.50,628.50,21.01,0.0290,894.7,230.3,40.0,700.855,...,682.940000,NaN,NaN,NaN,NaN,NaN,NaN,711.994131,0.009606,0.685595
2,2022-01-03,WELL-01,767.14,709.38,24.65,0.0311,924.7,284.0,40.0,700.855,...,711.006667,NaN,NaN,NaN,NaN,NaN,NaN,711.994131,0.009606,0.685595
3,2022-01-04,WELL-01,693.20,642.66,37.42,0.0512,927.1,165.4,40.0,699.210,...,706.555000,NaN,NaN,NaN,NaN,NaN,NaN,711.994131,0.009606,0.685595
4,2022-01-05,WELL-01,699.21,618.51,18.57,0.0259,884.6,176.0,40.0,700.855,...,705.086000,NaN,NaN,NaN,NaN,NaN,NaN,711.994131,0.009606,0.685595


## 3. Classical Decline Curve Analysis — Arps Hyperbolic Fit

The industry-standard technique: fit `q(t) = qi / (1 + b·di·t)^(1/b)` per well using non-linear least squares,
withholding the last 90 days as a **fair holdout** to score forecast accuracy, and integrating the curve out to
an economic limit to estimate **EUR (Estimated Ultimate Recovery)**.

In [4]:
import duckdb
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

con = duckdb.connect("dca_warehouse.duckdb")
gold_df = con.execute("SELECT * FROM gold_features ORDER BY well_id, date").df()

def arps_hyperbolic(t, qi, di, b):
    b = max(b, 1e-6)
    return qi / np.power(1 + b * di * t, 1.0 / b)

def fit_arps(well_df, holdout_days=90):
    """Fit on all-but-last `holdout_days`, so we can score forecast accuracy honestly."""
    d = well_df[well_df["oil_rate_bbl"] > 0].reset_index(drop=True)
    if len(d) < holdout_days + 60:
        return None

    train = d.iloc[: -holdout_days]
    test = d.iloc[-holdout_days:]

    t_train = train["days_on_prod"].values.astype(float)
    q_train = train["oil_rate_bbl"].values.astype(float)

    try:
        popt, _ = curve_fit(
            arps_hyperbolic, t_train, q_train,
            p0=[q_train[0], 0.005, 0.5],
            bounds=([1, 1e-5, 1e-4], [q_train.max() * 2, 0.05, 1.0]),
            maxfev=8000,
        )
    except Exception:
        return None

    qi, di, b = popt
    t_test = test["days_on_prod"].values.astype(float)
    pred_test = arps_hyperbolic(t_test, qi, di, b)
    rmse = np.sqrt(np.mean((pred_test - test["oil_rate_bbl"].values) ** 2))

    # EUR: integrate rate out to an economic limit of 10 bbl/d or 30 years, whichever first
    t_future = np.arange(0, 365 * 30)
    q_future = arps_hyperbolic(t_future, qi, di, b)
    econ_limit_idx = np.argmax(q_future < 10) if np.any(q_future < 10) else len(q_future)
    trapz_fn = getattr(np, "trapezoid", None) or np.trapz
    eur = trapz_fn(q_future[:econ_limit_idx], t_future[:econ_limit_idx])

    return {
        "well_id": well_df["well_id"].iloc[0],
        "qi_fit": qi, "di_fit": di, "b_fit": b,
        "holdout_rmse": rmse, "eur_bbl": eur,
        "last_day": d["days_on_prod"].max(),
    }

arps_results, arps_forecast_rows = [], []
for wid, g in gold_df.groupby("well_id"):
    res = fit_arps(g)
    if res is None:
        continue
    arps_results.append(res)
    t_fc = np.arange(0, res["last_day"] + 366)
    q_fc = arps_hyperbolic(t_fc, res["qi_fit"], res["di_fit"], res["b_fit"])
    for t, q in zip(t_fc, q_fc):
        arps_forecast_rows.append({"well_id": wid, "days_on_prod": int(t), "arps_rate_bbl": q})

arps_params_df = pd.DataFrame(arps_results)
arps_forecast_df = pd.DataFrame(arps_forecast_rows)

con.execute("CREATE OR REPLACE TABLE arps_params AS SELECT * FROM arps_params_df")
con.execute("CREATE OR REPLACE TABLE arps_forecast AS SELECT * FROM arps_forecast_df")
con.close()

print(f"Fit classical Arps hyperbolic decline for {len(arps_params_df)} wells")
print(f"Average 90-day holdout RMSE (classical DCA): {arps_params_df['holdout_rmse'].mean():.2f} bbl/d")
arps_params_df[["well_id", "qi_fit", "di_fit", "b_fit", "holdout_rmse", "eur_bbl"]].round(3)


Fit classical Arps hyperbolic decline for 15 wells
Average 90-day holdout RMSE (classical DCA): 4.75 bbl/d


,well_id,qi_fit,di_fit,b_fit,holdout_rmse,eur_bbl
0,WELL-01,725.566,0.010,0.696,2.226,176038.263
1,WELL-02,1047.601,0.008,0.108,0.516,141941.884
2,WELL-03,1130.800,0.007,0.701,5.441,438502.432
3,WELL-04,374.891,0.004,0.358,1.619,119413.293
4,WELL-05,1156.240,0.002,0.923,18.747,1554917.693
5,WELL-06,684.760,0.003,0.571,6.214,382756.810
6,WELL-07,810.556,0.007,0.340,1.480,158651.756
7,WELL-08,1057.661,0.002,0.545,15.286,943557.733
8,WELL-09,797.944,0.004,0.183,2.954,241292.074
9,WELL-10,482.765,0.008,0.087,0.213,61734.054


## 4. AI Layer — Global XGBoost Forecasting Model

Instead of fitting each well in isolation (as classical Arps does), we train **one model across all wells**
using engineered features (rolling averages, decline rate, water cut/GOR trends, lag features). This lets the
model borrow strength across wells and capture non-Arps behavior (e.g. water breakthrough effects) that a
pure physics-based decline curve can't represent. We benchmark it against Arps on the same 90-day holdout.

In [5]:
import duckdb
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

con = duckdb.connect("dca_warehouse.duckdb")
gold_df = con.execute("SELECT * FROM gold_features ORDER BY well_id, date").df()
arps_params_df = con.execute("SELECT * FROM arps_params").df()

FEATURES = [
    "days_on_prod", "rate_7d_avg", "rate_30d_avg", "decline_rate_pct",
    "water_cut", "water_cut_slope_30d", "gor_scf_bbl", "gor_slope_30d",
    "wellhead_pressure_psi", "choke_size_64th",
    "oil_rate_lag7", "oil_rate_lag14", "oil_rate_lag30",
]
TARGET = "oil_rate_bbl"
HOLDOUT_DAYS = 90

model_df = gold_df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + [TARGET]).copy()

train_frames, test_frames = [], []
for wid, g in model_df.groupby("well_id"):
    g = g.sort_values("days_on_prod")
    if len(g) < HOLDOUT_DAYS + 60:
        continue
    train_frames.append(g.iloc[:-HOLDOUT_DAYS])
    test_frames.append(g.iloc[-HOLDOUT_DAYS:])

train_df = pd.concat(train_frames)
test_df = pd.concat(test_frames)

# one global model trained ACROSS all wells -> this is the "AI adds value beyond single-well curve fitting" story
model = XGBRegressor(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
)
model.fit(train_df[FEATURES], train_df[TARGET])

test_df = test_df.copy()
test_df["ml_pred"] = model.predict(test_df[FEATURES]).clip(min=0)
ml_rmse_by_well = (
    test_df.groupby("well_id")
    .apply(lambda g: np.sqrt(mean_squared_error(g[TARGET], g["ml_pred"])))
    .rename("ml_holdout_rmse")
    .reset_index()
)

comparison = arps_params_df.merge(ml_rmse_by_well, on="well_id", how="inner")
comparison["ml_better"] = comparison["ml_holdout_rmse"] < comparison["holdout_rmse"]

print(f"Global XGBoost model trained on {len(train_df):,} well-days across {train_df['well_id'].nunique()} wells")
print(f"Avg 90-day holdout RMSE  ->  Arps: {comparison['holdout_rmse'].mean():.2f} bbl/d   "
      f"|   XGBoost: {comparison['ml_holdout_rmse'].mean():.2f} bbl/d")
print(f"XGBoost beat classical Arps on {comparison['ml_better'].sum()}/{len(comparison)} wells")

# feature importance -> useful, explainable AI output for the report/app
importance_df = (
    pd.Series(model.feature_importances_, index=FEATURES)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "feature", 0: "importance"})
)

# persist model predictions (test window) + comparison table + feature importance
con.execute("CREATE OR REPLACE TABLE ml_predictions AS SELECT * FROM test_df")
con.execute("CREATE OR REPLACE TABLE model_comparison AS SELECT * FROM comparison")
con.execute("CREATE OR REPLACE TABLE feature_importance AS SELECT * FROM importance_df")

import pickle
with open("xgb_forecast_model.pkl", "wb") as f:
    pickle.dump({"model": model, "features": FEATURES}, f)

con.close()
comparison[["well_id", "holdout_rmse", "ml_holdout_rmse", "ml_better"]].round(2)


/tmp/ipykernel_3406/908951050.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.sqrt(mean_squared_error(g[TARGET], g["ml_pred"])))


Global XGBoost model trained on 11,593 well-days across 15 wells
Avg 90-day holdout RMSE  ->  Arps: 4.75 bbl/d   |   XGBoost: 5.51 bbl/d
XGBoost beat classical Arps on 3/15 wells


,well_id,holdout_rmse,ml_holdout_rmse,ml_better
0,WELL-01,2.23,2.49,False
1,WELL-02,0.52,1.48,False
2,WELL-03,5.44,6.34,False
3,WELL-04,1.62,1.93,False
4,WELL-05,18.75,18.33,True
5,WELL-06,6.21,5.94,True
6,WELL-07,1.48,2.38,False
7,WELL-08,15.29,15.26,True
8,WELL-09,2.95,3.54,False
9,WELL-10,0.21,3.20,False


## 5. AI Layer — Anomaly Detection

We compute each well's **residual** (actual rate vs. what its classical Arps curve predicts) and feed it,
along with water cut and GOR trend slopes, into an **Isolation Forest**. This flags wells that are deviating
from their natural decline — a sign of mechanical issues, early water/gas breakthrough, or metering problems —
something a plain decline-curve project never surfaces.

In [6]:
import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

con = duckdb.connect("dca_warehouse.duckdb")
gold_df = con.execute("SELECT * FROM gold_features ORDER BY well_id, date").df()
arps_forecast_df = con.execute("SELECT * FROM arps_forecast").df()

# residual = actual rate vs what the classical Arps curve expects -> a well drifting far
# below its natural decline trend is a mechanical/operational red flag, not just "decline"
merged = gold_df.merge(arps_forecast_df, on=["well_id", "days_on_prod"], how="inner")
merged["residual"] = merged["oil_rate_bbl"] - merged["arps_rate_bbl"]
merged["residual_pct"] = merged["residual"] / merged["arps_rate_bbl"].replace(0, np.nan) * 100

ANOMALY_FEATURES = ["residual_pct", "water_cut_slope_30d", "gor_slope_30d", "decline_rate_pct"]
anom_input = merged.dropna(subset=ANOMALY_FEATURES).copy()

iso = IsolationForest(n_estimators=300, contamination=0.03, random_state=42)
anom_input["anomaly_score"] = iso.fit_predict(anom_input[ANOMALY_FEATURES])
anom_input["is_anomaly"] = anom_input["anomaly_score"] == -1

anomalies = anom_input[anom_input["is_anomaly"]][
    ["well_id", "date", "oil_rate_bbl", "arps_rate_bbl", "residual_pct",
     "water_cut_slope_30d", "gor_slope_30d"]
].sort_values("date")

def classify_anomaly(row):
    if row["residual_pct"] < -20 and row["gor_slope_30d"] > 1:
        return "Possible mechanical failure / gas breakthrough"
    if row["water_cut_slope_30d"] > 0.005:
        return "Rapid water breakthrough"
    if row["residual_pct"] < -20:
        return "Underperforming vs natural decline"
    if row["residual_pct"] > 20:
        return "Outperforming forecast (verify metering)"
    return "Irregular production pattern"

anomalies["likely_cause"] = anomalies.apply(classify_anomaly, axis=1)

con.execute("CREATE OR REPLACE TABLE anomalies AS SELECT * FROM anomalies")
con.close()

print(f"Flagged {len(anomalies)} anomalous well-days out of {len(anom_input):,} scored ({len(anomalies)/len(anom_input)*100:.1f}%)")
print(f"Wells affected: {anomalies['well_id'].nunique()}")
anomalies.head(15).round(2)


Flagged 389 anomalous well-days out of 12,943 scored (3.0%)
Wells affected: 15


,well_id,date,oil_rate_bbl,arps_rate_bbl,residual_pct,water_cut_slope_30d,gor_slope_30d,likely_cause
10841,WELL-13,2022-02-11,283.14,315.75,-10.33,-0.0,2.74,Irregular production pattern
5443,WELL-07,2022-02-13,641.76,601.13,6.76,0.0,-2.18,Irregular production pattern
5469,WELL-07,2022-03-11,542.36,508.66,6.62,0.0,-1.59,Irregular production pattern
1892,WELL-03,2022-04-03,755.10,685.06,10.22,0.0,-2.61,Irregular production pattern
4626,WELL-06,2022-05-07,0.00,462.63,-100.00,0.0,-1.71,Underperforming vs natural decline
4627,WELL-06,2022-05-08,0.00,461.34,-100.00,0.0,-0.02,Underperforming vs natural decline
4628,WELL-06,2022-05-09,0.00,460.06,-100.00,-0.0,1.01,Possible mechanical failure / gas breakthrough
4629,WELL-06,2022-05-10,0.00,458.79,-100.00,0.0,1.02,Possible mechanical failure / gas breakthrough
4630,WELL-06,2022-05-11,0.00,457.52,-100.00,0.0,1.19,Possible mechanical failure / gas breakthrough
4631,WELL-06,2022-05-12,0.00,456.25,-100.00,0.0,1.96,Possible mechanical failure / gas breakthrough


## 6. Streamlit Dashboard

Writes `app.py` to disk, multi-tab dashboard with:
- Well selector + KPI cards (qi, b-factor, EUR, RMSE comparison)
- Production forecast chart: actual vs. classical Arps vs. XGBoost, with anomalies marked
- Water cut & GOR trend charts
- Log-log and rate-cumulative diagnostic plots
- Anomaly table with likely-cause classification
- Field-wide model comparison and feature importance

In [11]:
%%writefile app.py
import duckdb
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
from plotly.subplots import make_subplots

st.set_page_config(page_title="Smart DCA Dashboard", page_icon="🛢️", layout="wide")

# ---------------------------------------------------------------- palette
# Modified for a light/white background with high-contrast text and warm oil accents.
OIL_BLACK   = "#1a1310"
RUST        = "#c2410c"
AMBER       = "#d97706"  # Darkened slightly for better readability on white
COPPER      = "#b45309"
OLIVE       = "#60643a"  # Darkened for text contrast
BG_WHITE    = "#ffffff"
TEXT_DARK   = "#1f2937"  # Dark gray for crisp body text readability
BORDER_GRAY = "#e5e7eb"
DEEP_RUST   = "#7c2d12"

st.markdown(f"""
<style>
    :root {{
        --oil-black: {OIL_BLACK};
        --rust: {RUST};
        --amber: {AMBER};
        --copper: {COPPER};
        --text-dark: {TEXT_DARK};
        --bg-white: {BG_WHITE};
        --border-gray: {BORDER_GRAY};
    }}

    .stApp {{
        background: {BG_WHITE};
        color: var(--text-dark) !important;
    }}
    html, body, [class*="css"] {{ color: var(--text-dark) !important; }}

    /* Clear header backgrounds */
    header[data-testid="stHeader"] {{
        background-color: transparent !important;
        background: transparent !important;
    }}
    div[data-testid="stDecoration"] {{ display: none !important; }}
    header[data-testid="stHeader"] * {{ color: var(--text-dark) !important; fill: var(--text-dark) !important; }}

    h1, h2, h3, h4 {{ color: var(--copper) !important; font-weight: 800 !important; }}
    p, span, label, div {{ color: var(--text-dark); }}

    /* Light sidebar styling */
    section[data-testid="stSidebar"] {{
        background: #f9fafb;
        border-right: 1px solid var(--border-gray);
    }}
    section[data-testid="stSidebar"] * {{ color: var(--text-dark) !important; }}

    .stSelectbox label, .stCheckbox label, .stCaption, [data-testid="stCaptionContainer"] {{
        color: var(--text-dark) !important;
        opacity: 0.9;
    }}

    /* Selectbox light theme adjustments */
    div[data-baseweb="select"] {{ background-color: #ffffff !important; }}
    div[data-baseweb="select"] > div {{
        background-color: #ffffff !important;
        border-color: var(--border-gray) !important;
        color: var(--text-dark) !important;
    }}
    div[data-baseweb="select"] * {{ color: var(--text-dark) !important; background-color: transparent !important; }}
    div[data-baseweb="select"] svg {{ fill: var(--text-dark) !important; }}
    ul[data-testid="stSelectboxVirtualDropdown"] {{ background-color: #ffffff !important; border: 1px solid var(--border-gray); }}
    ul[data-testid="stSelectboxVirtualDropdown"] li {{ background-color: #ffffff !important; color: var(--text-dark) !important; }}
    ul[data-testid="stSelectboxVirtualDropdown"] li:hover {{ background-color: #f3f4f6 !important; }}

    /* Tabs styling */
    button[data-baseweb="tab"] {{
        color: var(--text-dark) !important;
        font-weight: 600;
    }}
    button[data-baseweb="tab"][aria-selected="true"] {{
        color: var(--amber) !important;
        border-bottom: 3px solid var(--rust) !important;
    }}
    div[data-baseweb="tab-highlight"] {{ background-color: var(--rust) !important; }}

    [data-testid="stAppViewContainer"] > .main .block-container {{
        padding-top: 1.5rem !important;
    }}
    button[data-testid="stSidebarCollapseButton"], button[data-testid="baseButton-headerNoPadding"] {{
        color: var(--text-dark) !important;
    }}
    button[data-testid="stSidebarCollapseButton"] svg {{ fill: var(--text-dark) !important; }}

    /* Metric cards: Clean, light style with crisp borders */
    .metric-card {{
        background: #ffffff;
        border: 1px solid var(--border-gray);
        border-radius: 14px;
        padding: 18px 20px;
        text-align: center;
        box-shadow: 0 4px 12px rgba(0,0,0,0.05);
    }}
    .metric-card h3 {{ color: #6b7280 !important; font-size: 12.5px; font-weight: 600;
                       margin: 0 0 6px 0; text-transform: uppercase; letter-spacing: .6px;}}
    .metric-card p {{ color: var(--oil-black) !important; font-size: 27px; font-weight: 800; margin: 0; }}
    .badge-good {{ background:#dcfce7; color:#15803d; padding:3px 10px; border-radius:20px; font-size:12px; font-weight: 600;}}
    .badge-bad  {{ background:#fee2e2; color:#b91c1c; padding:3px 10px; border-radius:20px; font-size:12px; font-weight: 600;}}

    [data-testid="stDataFrame"] {{ border: 1px solid var(--border-gray); border-radius: 10px; }}

    hr {{ border-color: var(--border-gray) !important; }}

    /* Animated banner shifted to a crisp light layout */
    .rig-banner {{
        position: relative;
        height: 130px;
        margin-bottom: 6px;
        overflow: hidden;
        border-radius: 14px;
        background: linear-gradient(180deg, #f9fafb 0%, #f3f4f6 100%);
        border: 1px solid var(--border-gray);
    }}
    .rig-banner svg {{ position: absolute; bottom: 0; left: 0; }}
    .pumpjack-arm {{
        transform-origin: 140px 55px;
        animation: pump 2.4s ease-in-out infinite;
    }}
    @keyframes pump {{
        0%   {{ transform: rotate(-14deg); }}
        50%  {{ transform: rotate(14deg); }}
        100% {{ transform: rotate(-14deg); }}
    }}
    .pump-head {{
        transform-origin: 140px 55px;
        animation: pumphead 2.4s ease-in-out infinite;
    }}
    @keyframes pumphead {{
        0%   {{ transform: translateY(-6px); }}
        50%  {{ transform: translateY(6px); }}
        100% {{ transform: translateY(-6px); }}
    }}
    .oil-drop {{
        position: absolute;
        bottom: 100%;
        font-size: 20px;
        opacity: 0.85;
        animation: drop linear infinite;
    }}
    @keyframes drop {{
        0%   {{ transform: translateY(-10px); opacity: 0; }}
        15%  {{ opacity: 0.9; }}
        100% {{ transform: translateY(150px); opacity: 0; }}
    }}
</style>
""", unsafe_allow_html=True)

# ---------------------------------------------------------------- animated banner
st.markdown(f"""
<div class="rig-banner">
  <svg width="100%" height="130" viewBox="0 0 900 130" preserveAspectRatio="xMidYMax slice">
    <rect x="0" y="118" width="900" height="12" fill="{RUST}" opacity="0.2"/>
    <g opacity="0.4" stroke="{COPPER}" stroke-width="3" fill="none">
      <path d="M700 118 L710 55 L720 118" />
      <path d="M703 85 L717 85" />
      <path d="M705 100 L715 100" />
      <path d="M760 118 L768 65 L776 118" />
      <path d="M762 90 L774 90" />
    </g>
    <g transform="translate(60,0)">
      <rect x="20" y="108" width="16" height="10" fill="{RUST}"/>
      <line x1="28" y1="108" x2="28" y2="55" stroke="{COPPER}" stroke-width="5"/>
      <line x1="10" y1="112" x2="46" y2="60" stroke="{COPPER}" stroke-width="4"/>
      <g class="pumpjack-arm">
        <line x1="0" y1="55" x2="80" y2="55" stroke="{AMBER}" stroke-width="6" stroke-linecap="round"/>
        <circle cx="0" cy="55" r="6" fill="{RUST}"/>
      </g>
      <g class="pump-head">
        <line x1="0" y1="55" x2="0" y2="100" stroke="{AMBER}" stroke-width="4"/>
      </g>
      <circle cx="28" cy="55" r="7" fill="{DEEP_RUST}"/>
    </g>
    <g transform="translate(830,88)" opacity="0.9">
      <rect x="0" y="0" width="28" height="30" rx="4" fill="{RUST}"/>
      <ellipse cx="14" cy="0" rx="14" ry="4" fill="{AMBER}"/>
      <text x="6" y="20" font-size="10" fill="#ffffff" font-weight="bold">OIL</text>
    </g>
  </svg>
  <span class="oil-drop" style="left:145px; animation-duration:2.6s;">🛢️</span>
  <span class="oil-drop" style="left:200px; animation-duration:3.4s; animation-delay:.8s; font-size:14px;">●</span>
  <span class="oil-drop" style="left:170px; animation-duration:3s; animation-delay:1.6s; font-size:12px;">●</span>
</div>
""", unsafe_allow_html=True)

@st.cache_resource
def get_con():
    return duckdb.connect("dca_warehouse.duckdb", read_only=True)

con = get_con()

@st.cache_data
def load_tables():
    gold = con.execute("SELECT * FROM gold_features ORDER BY well_id, date").df()
    arps_params = con.execute("SELECT * FROM arps_params").df()
    arps_forecast = con.execute("SELECT * FROM arps_forecast").df()
    ml_pred = con.execute("SELECT * FROM ml_predictions").df()
    comparison = con.execute("SELECT * FROM model_comparison").df()
    anomalies = con.execute("SELECT * FROM anomalies").df()
    importance = con.execute("SELECT * FROM feature_importance").df()
    meta = con.execute("SELECT * FROM well_metadata").df()
    return gold, arps_params, arps_forecast, ml_pred, comparison, anomalies, importance, meta

gold, arps_params, arps_forecast, ml_pred, comparison, anomalies, importance, meta = load_tables()

# ---------------------------------------------------------------- sidebar
st.sidebar.markdown("## 🛢️ Smart DCA Dashboard")
st.sidebar.caption("Decline Curve Analysis + Data Engineering + AI")
wells = sorted(gold["well_id"].unique())
well = st.sidebar.selectbox("Select Well", wells)
st.sidebar.markdown("---")
show_ml = st.sidebar.checkbox("Show ML (XGBoost) forecast", value=True)
show_arps = st.sidebar.checkbox("Show classical Arps forecast", value=True)
show_anomalies = st.sidebar.checkbox("Highlight anomalies", value=True)
st.sidebar.markdown("---")
st.sidebar.markdown("**Pipeline layers**")
st.sidebar.markdown("`bronze` → raw SCADA export\n\n`silver` → cleaned & gap-filled\n\n`gold` → feature-engineered")

well_meta = meta[meta["well_id"] == well].iloc[0]
well_gold = gold[gold["well_id"] == well].sort_values("date")
well_arps_params = arps_params[arps_params["well_id"] == well]
well_arps_fc = arps_forecast[arps_forecast["well_id"] == well]
well_ml = ml_pred[ml_pred["well_id"] == well]
well_comp = comparison[comparison["well_id"] == well]
well_anom = anomalies[anomalies["well_id"] == well]

# ---------------------------------------------------------------- header
st.markdown(f"# 🛢️ {well}")
st.caption(f"Spud date {well_meta['spud_date']}  |  Choke {int(well_meta['choke_size_64th'])}/64\"  |  "
           f"Lat {well_meta['latitude']:.3f}, Lon {well_meta['longitude']:.3f}")

if len(well_arps_params):
    p = well_arps_params.iloc[0]
    c1, c2, c3, c4, c5 = st.columns(5)
    with c1:
        st.markdown(f'<div class="metric-card"><h3>Initial Rate (qi)</h3><p>{p["qi_fit"]:,.0f} bbl/d</p></div>', unsafe_allow_html=True)
    with c2:
        st.markdown(f'<div class="metric-card"><h3>Decline (b-factor)</h3><p>{p["b_fit"]:.2f}</p></div>', unsafe_allow_html=True)
    with c3:
        st.markdown(f'<div class="metric-card"><h3>EUR (Arps)</h3><p>{p["eur_bbl"]/1000:,.0f}k bbl</p></div>', unsafe_allow_html=True)
    with c4:
        arps_rmse = p["holdout_rmse"]
        st.markdown(f'<div class="metric-card"><h3>Arps 90d RMSE</h3><p>{arps_rmse:.1f}</p></div>', unsafe_allow_html=True)
    with c5:
        if len(well_comp):
            ml_rmse = well_comp.iloc[0]["ml_holdout_rmse"]
            better = "badge-good" if ml_rmse < arps_rmse else "badge-bad"
            label = "ML wins" if ml_rmse < arps_rmse else "Arps wins"
            st.markdown(f'<div class="metric-card"><h3>XGBoost 90d RMSE</h3><p>{ml_rmse:.1f}</p>'
                        f'<span class="{better}">{label}</span></div>', unsafe_allow_html=True)

st.markdown("###")

tab1, tab2, tab3, tab4 = st.tabs(["📈 Production Forecast", "📉 Diagnostic (log-log)", "🚨 Anomalies", "🧠 Model Insights"])

# Plot templates flipped to a clear white/light configuration with light-gray grid lines
PLOT_TEMPLATE_LAYOUT = dict(
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font=dict(color=TEXT_DARK),
    xaxis=dict(gridcolor=BORDER_GRAY, zerolinecolor=BORDER_GRAY),
    yaxis=dict(gridcolor=BORDER_GRAY, zerolinecolor=BORDER_GRAY),
)

# ---------------------------------------------------------------- tab 1: forecast
with tab1:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=well_gold["date"], y=well_gold["oil_rate_bbl"],
        mode="lines", name="Actual (cleaned)", line=dict(color=AMBER, width=1.8)
    ))
    if show_arps and len(well_arps_fc):
        fc_dates = pd.to_datetime(well_gold["date"].iloc[0]) + pd.to_timedelta(well_arps_fc["days_on_prod"], unit="D")
        fig.add_trace(go.Scatter(
            x=fc_dates, y=well_arps_fc["arps_rate_bbl"],
            mode="lines", name="Classical Arps fit/forecast", line=dict(color=OLIVE, width=2, dash="dash")
        ))
    if show_ml and len(well_ml):
        fig.add_trace(go.Scatter(
            x=well_ml["date"], y=well_ml["ml_pred"],
            mode="markers", name="XGBoost forecast (holdout)", marker=dict(color="#b45309", size=6, symbol="diamond")
        ))
    if show_anomalies and len(well_anom):
        fig.add_trace(go.Scatter(
            x=well_anom["date"], y=well_anom["oil_rate_bbl"],
            mode="markers", name="Anomaly flagged", marker=dict(color=RUST, size=10, symbol="x")
        ))
    fig.update_layout(
        template=None, height=480, hovermode="x unified",
        margin=dict(l=10, r=10, t=30, b=10),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, font=dict(color=TEXT_DARK)),
        xaxis_title="Date", yaxis_title="Oil Rate (bbl/d)",
        **PLOT_TEMPLATE_LAYOUT,
    )
    st.plotly_chart(fig, use_container_width=True)

    c1, c2 = st.columns(2)
    with c1:
        st.markdown("**Water cut trend**")
        fig_wc = go.Figure(go.Scatter(x=well_gold["date"], y=well_gold["water_cut"]*100,
                                       line=dict(color=RUST), fill="tozeroy", fillcolor="rgba(194,65,12,0.12)"))
        fig_wc.update_layout(height=260, margin=dict(l=10,r=10,t=10,b=10),
                              yaxis_title="Water cut (%)", **PLOT_TEMPLATE_LAYOUT)
        st.plotly_chart(fig_wc, use_container_width=True)
    with c2:
        st.markdown("**GOR trend**")
        fig_gor = go.Figure(go.Scatter(x=well_gold["date"], y=well_gold["gor_scf_bbl"],
                                        line=dict(color=COPPER), fill="tozeroy", fillcolor="rgba(180,83,9,0.12)"))
        fig_gor.update_layout(height=260, margin=dict(l=10,r=10,t=10,b=10),
                               yaxis_title="GOR (scf/bbl)", **PLOT_TEMPLATE_LAYOUT)
        st.plotly_chart(fig_gor, use_container_width=True)

# ---------------------------------------------------------------- tab 2: diagnostic log-log
with tab2:
    st.markdown("Classical rate-cumulative and log-log diagnostic plots used to sanity-check the decline regime.")
    d = well_gold[well_gold["oil_rate_bbl"] > 0]
    fig_diag = make_subplots(rows=1, cols=2, subplot_titles=("Rate vs Time (log-log)", "Rate vs Cumulative"))
    fig_diag.add_trace(go.Scatter(x=d["days_on_prod"]+1, y=d["oil_rate_bbl"], mode="markers",
                                   marker=dict(color=AMBER, size=4)), row=1, col=1)
    fig_diag.update_xaxes(type="log", title_text="Days on production", row=1, col=1)
    fig_diag.update_yaxes(type="log", title_text="Oil rate (bbl/d)", row=1, col=1)

    fig_diag.add_trace(go.Scatter(x=d["cum_oil_bbl"], y=d["oil_rate_bbl"], mode="markers",
                                   marker=dict(color=RUST, size=4)), row=1, col=2)
    fig_diag.update_xaxes(title_text="Cumulative oil (bbl)", row=1, col=2)
    fig_diag.update_yaxes(title_text="Oil rate (bbl/d)", row=1, col=2)

    fig_diag.update_layout(height=420, showlegend=False, **PLOT_TEMPLATE_LAYOUT)
    fig_diag.update_annotations(font=dict(color=TEXT_DARK))
    st.plotly_chart(fig_diag, use_container_width=True)

# ---------------------------------------------------------------- tab 3: anomalies
with tab3:
    st.markdown(f"### {len(well_anom)} anomalous readings detected for {well}")
    if len(well_anom):
        st.dataframe(
            well_anom[["date", "oil_rate_bbl", "arps_rate_bbl", "residual_pct", "likely_cause"]]
            .rename(columns={
                "date": "Date", "oil_rate_bbl": "Actual (bbl/d)", "arps_rate_bbl": "Expected/Arps (bbl/d)",
                "residual_pct": "Residual (%)", "likely_cause": "Likely Cause"
            }).round(1),
            use_container_width=True, height=350
        )
    else:
        st.success("No anomalies flagged for this well — production tracking its natural decline trend.")

    st.markdown("### Field-wide anomaly summary")
    field_summary = anomalies.groupby("well_id").size().reset_index(name="anomaly_count").sort_values("anomaly_count", ascending=False)
    fig_field = go.Figure(go.Bar(x=field_summary["well_id"], y=field_summary["anomaly_count"], marker_color=RUST))
    fig_field.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10),
                             xaxis_title="Well", yaxis_title="Anomalous days", **PLOT_TEMPLATE_LAYOUT)
    st.plotly_chart(fig_field, use_container_width=True)

# ---------------------------------------------------------------- tab 4: model insights
with tab4:
    st.markdown("### Arps vs XGBoost — field-wide holdout accuracy")
    fig_comp = go.Figure()
    fig_comp.add_trace(go.Bar(x=comparison["well_id"], y=comparison["holdout_rmse"], name="Arps RMSE", marker_color=OLIVE))
    fig_comp.add_trace(go.Bar(x=comparison["well_id"], y=comparison["ml_holdout_rmse"], name="XGBoost RMSE", marker_color=AMBER))
    fig_comp.update_layout(barmode="group", height=380,
                            yaxis_title="90-day holdout RMSE (bbl/d)",
                            legend=dict(font=dict(color=TEXT_DARK)), **PLOT_TEMPLATE_LAYOUT)
    st.plotly_chart(fig_comp, use_container_width=True)

    win_rate = (comparison["ml_holdout_rmse"] < comparison["holdout_rmse"]).mean() * 100
    st.info(f"XGBoost outperformed the classical Arps fit on **{win_rate:.0f}%** of wells (lower 90-day holdout RMSE).")

    st.markdown("### What drives the ML forecast? (feature importance)")
    fig_imp = go.Figure(go.Bar(x=importance["importance"], y=importance["feature"], orientation="h", marker_color=COPPER))
    fig_imp.update_layout(height=420, margin=dict(l=10,r=10,t=10,b=10),
                           yaxis=dict(autorange="reversed", gridcolor=BORDER_GRAY), xaxis_title="Importance",
                           **{k: v for k, v in PLOT_TEMPLATE_LAYOUT.items() if k != "yaxis"})
    st.plotly_chart(fig_imp, use_container_width=True)

st.markdown("---")
st.caption("Bronze → Silver → Gold pipeline in DuckDB · Classical Arps (SciPy) vs XGBoost · Isolation Forest anomaly detection · Built with Streamlit")

Overwriting app.py


## 7. Launch the app from Colab

Get a **free** authtoken from https://dashboard.ngrok.com/get-started/your-authtoken (sign up takes 30 seconds),
paste it into the cell below, then run it. It will print a public URL you can open directly.

In [12]:
from pyngrok import ngrok, conf
import subprocess, time

# 1) Get a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
#    and paste it below (only needs to be run once per Colab session).
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# kill any previous tunnels/streamlit processes from earlier reruns in THIS session
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"]
)
time.sleep(6)

# Free ngrok accounts only allow ONE active tunnel at a time. If a previous Colab
# session disconnected without cleanly closing its tunnel, ngrok's servers may still
# think that old tunnel is "online" (ERR_NGROK_334) even though ngrok.kill() above
# only stops the LOCAL process. Retry a few times, since these stale tunnels usually
# expire on their own within a minute or two.
public_url = None
for attempt in range(5):
    try:
        public_url = ngrok.connect(8501)
        break
    except Exception as e:
        if "ERR_NGROK_334" in str(e) or "already online" in str(e):
            print(f"Attempt {attempt+1}/5: a previous tunnel is still live on ngrok's side, retrying in 15s...")
            print("If this keeps failing, go to https://dashboard.ngrok.com/endpoints and click 'Stop' "
                  "on the online endpoint, then rerun this cell.")
            time.sleep(15)
        else:
            raise

if public_url:
    print("🚀 Your Smart DCA Dashboard is live at:")
    print(public_url)
else:
    print("❌ Could not start the tunnel after 5 attempts.")
    print("Go to https://dashboard.ngrok.com/endpoints, stop the online endpoint, then rerun this cell.")


🚀 Your Smart DCA Dashboard is live at:
NgrokTunnel: "https://handcraft-keep-walmart.ngrok-free.dev" -> "http://localhost:8501"
